# High-price interval calibration

Issue #20: investigate intervals for the fixed county-verified Poisson model. Fit only on 2020-2023 training sales; calibrate on January-June 2024; compare methods on July-December 2024. The 2025 split is not read into any interval calculation here. Training-price bands are diagnostic only: a future sale's price is unknown at inference time.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from homelens.modeling.baseline import _read_audit, _validate_cohort
from homelens.modeling.county_comparison import _fit, _predict

cohort_path = Path(os.environ.get('HOMELENS_COHORT_PATH', 'data/processed/modeling_cohort_county_verified.csv'))
audit_path = Path(os.environ.get('HOMELENS_COHORT_AUDIT_PATH', 'data/processed/modeling_cohort_county_verified_audit.json'))
validation_start, test_start, audit = _read_audit(audit_path)
cohort = pd.read_csv(cohort_path, dtype={'zip': 'string', 'property_type': 'string', 'split': 'string'})
frame = _validate_cohort(cohort, validation_start, test_start)
assert len(frame) == audit['prepared_rows']
train = frame.loc[frame['split'].eq('train')].copy()
validation = frame.loc[frame['split'].eq('validation')].copy()
model, categories = _fit(frame)
prediction = _predict(model, categories, validation)
train_prediction = _predict(model, categories, train)
actual = validation['price_usd'].to_numpy(dtype=float)
calibration = (pd.to_datetime(validation['sale_date']) < pd.Timestamp('2024-07-01')).to_numpy()
assessment = ~calibration
train_p90 = float(train['price_usd'].quantile(0.9))
print({'train_rows': len(train), 'calibration_rows': int(calibration.sum()), 'assessment_rows': int(assessment.sum()), 'train_p90_usd': round(train_p90, 2)})

{'train_rows': 13102, 'calibration_rows': 1514, 'assessment_rows': 1414, 'train_p90_usd': 595000.0}


The quantile uses the finite-sample conformal rank. Relative widths scale with the predicted price, which is available at inference time. The segmented candidate uses training-prediction median and p90 boundaries, not observed sale-price bands.

In [2]:
def conformal_quantile(values, coverage):
    ordered = np.sort(np.asarray(values, dtype=float))
    rank = min(len(ordered), int(np.ceil((len(ordered) + 1) * coverage)))
    return float(ordered[rank - 1])

errors = actual - prediction
cal = calibration
q_abs = conformal_quantile(np.abs(errors[cal]), 0.9)
q_relative = conformal_quantile(np.abs(errors[cal]) / prediction[cal], 0.9)
q_lower_abs = conformal_quantile(np.maximum(-errors[cal], 0), 0.95)
q_upper_abs = conformal_quantile(np.maximum(errors[cal], 0), 0.95)
q_lower_relative = conformal_quantile(np.maximum(-errors[cal], 0) / prediction[cal], 0.95)
q_upper_relative = conformal_quantile(np.maximum(errors[cal], 0) / prediction[cal], 0.95)
predicted_boundaries = np.quantile(train_prediction, [0.5, 0.9])
predicted_band = np.digitize(prediction, predicted_boundaries, right=True)
band_rows = {band: int((cal & (predicted_band == band)).sum()) for band in range(3)}
band_relative = {
    band: conformal_quantile(
        np.abs(errors[cal & (predicted_band == band)]) / prediction[cal & (predicted_band == band)], 0.9
    ) if band_rows[band] >= 30 else q_relative
    for band in range(3)
}
band_scale = np.array([band_relative[band] for band in predicted_band])
intervals = {
    'global_absolute': (prediction - q_abs, prediction + q_abs),
    'global_relative': (prediction * (1 - q_relative), prediction * (1 + q_relative)),
    'asymmetric_absolute': (prediction - q_lower_abs, prediction + q_upper_abs),
    'asymmetric_relative': (prediction * (1 - q_lower_relative), prediction * (1 + q_upper_relative)),
    'predicted_band_relative': (prediction * (1 - band_scale), prediction * (1 + band_scale)),
}
print({'predicted_band_calibration_rows': band_rows, 'train_prediction_boundaries_usd': predicted_boundaries.round(2).tolist()})

{'predicted_band_calibration_rows': {0: 696, 1: 644, 2: 174}, 'train_prediction_boundaries_usd': [350726.34, 562663.69]}


In [3]:
high = actual > train_p90
results = []
for name, (lower, upper) in intervals.items():
    covered = (actual >= lower) & (actual <= upper)
    results.append({
        'method': name,
        'assessment_rows': int(assessment.sum()),
        'coverage': round(float(covered[assessment].mean()), 4),
        'high_price_rows': int((assessment & high).sum()),
        'high_price_coverage': round(float(covered[assessment & high].mean()), 4),
        'median_width_usd': round(float(np.median((upper - lower)[assessment])), 2),
        'high_price_median_width_usd': round(float(np.median((upper - lower)[assessment & high])), 2),
    })
pd.DataFrame(results)

,method,assessment_rows,coverage,high_price_rows,high_price_coverage,median_width_usd,high_price_median_width_usd
0,global_absolute,1414,0.8975,219,0.4977,301116.57,301116.57
1,global_relative,1414,0.8996,219,0.6986,276588.46,458126.15
2,asymmetric_absolute,1414,0.9066,219,0.6347,285731.98,285731.98
3,asymmetric_relative,1414,0.9088,219,0.8265,247127.45,409328.52
4,predicted_band_relative,1414,0.9038,219,0.7626,265563.36,650661.88


A high-price miss is mainly an underprediction. Compare three fixed upper-tail allocations within a nominal 10% two-sided error budget. Selection still uses only the second half of 2024; the 2025 split remains untouched.

In [4]:
directional = []
for upper_tail in (0.025, 0.01, 0.005):
    lower_tail = 0.10 - upper_tail
    lower_scale = conformal_quantile(np.maximum(-errors[cal], 0) / prediction[cal], 1 - lower_tail)
    upper_scale = conformal_quantile(np.maximum(errors[cal], 0) / prediction[cal], 1 - upper_tail)
    lower = np.maximum(0, prediction * (1 - lower_scale))
    upper = prediction * (1 + upper_scale)
    covered = (actual >= lower) & (actual <= upper)
    directional.append({
        'upper_tail': upper_tail,
        'lower_tail': lower_tail,
        'coverage': round(float(covered[assessment].mean()), 4),
        'high_price_coverage': round(float(covered[assessment & high].mean()), 4),
        'median_width_usd': round(float(np.median((upper - lower)[assessment])), 2),
        'high_price_median_width_usd': round(float(np.median((upper - lower)[assessment & high])), 2),
        'upper_scale': round(upper_scale, 4),
        'lower_scale': round(lower_scale, 4),
    })
pd.DataFrame(directional)

,upper_tail,lower_tail,coverage,high_price_coverage,median_width_usd,high_price_median_width_usd,upper_scale,lower_scale
0,0.025,0.075,0.8982,0.8767,286826.64,475084.12,0.6362,0.1353
1,0.010,0.090,0.8939,0.9041,340288.67,563635.73,0.7961,0.1192
2,0.005,0.095,0.8953,0.9269,389025.52,644360.79,0.9308,0.1156


Freeze the narrowest directional candidate that reaches at least 88% overall and 90% above-training-p90 coverage on July-December 2024. That selects a 1% upper tail and 9% lower tail. This is an exploratory selection because the 2024 period previously informed the point-model loss choice; 2025 is reserved for one final, unchanged assessment. An interval that is too broad or unstable by slice should remain out of inference.

In [5]:
selected_upper_tail = 0.01
selected_lower_tail = 0.09
selected_lower_scale = conformal_quantile(np.maximum(-errors[cal], 0) / prediction[cal], 1 - selected_lower_tail)
selected_upper_scale = conformal_quantile(np.maximum(errors[cal], 0) / prediction[cal], 1 - selected_upper_tail)
selected_lower = np.maximum(0, prediction * (1 - selected_lower_scale))
selected_upper = prediction * (1 + selected_upper_scale)
selected_covered = (actual >= selected_lower) & (actual <= selected_upper)
train_p50 = float(train['price_usd'].quantile(0.5))
scored = validation.loc[assessment, ['property_type', 'zip']].copy()
scored['price_band'] = np.where(actual[assessment] <= train_p50, 'at_or_below_train_p50', np.where(actual[assessment] <= train_p90, 'train_p50_to_p90', 'above_train_p90'))
scored['covered'] = selected_covered[assessment]
scored['width_usd'] = (selected_upper - selected_lower)[assessment]
scored['predicted_usd'] = prediction[assessment]
slice_rows = []
for field in ('price_band', 'property_type', 'zip'):
    for value, group in scored.groupby(field, sort=True):
        supported = len(group) >= 30
        slice_rows.append({
            'slice': field, 'value': str(value), 'rows': len(group), 'small_slice': not supported,
            'coverage': round(float(group['covered'].mean()), 4) if supported else None,
            'median_width_usd': round(float(group['width_usd'].median()), 2) if supported else None,
            'median_width_to_prediction': round(float(group['width_usd'].median() / group['predicted_usd'].median()), 3) if supported else None,
        })
pd.DataFrame(slice_rows)

,slice,value,rows,small_slice,coverage,median_width_usd,median_width_to_prediction
0,price_band,above_train_p90,219,False,0.9041,563635.73,0.915
1,price_band,at_or_below_train_p50,452,False,0.8274,251987.97,0.915
2,price_band,train_p50_to_p90,743,False,0.9314,371265.47,0.915
3,property_type,Condo/Co-op,49,False,0.9796,167378.56,0.915
4,property_type,Single Family Residential,1031,False,0.8739,368418.56,0.915
5,property_type,Townhouse,334,False,0.9431,302371.13,0.915
6,zip,27503,12,True,NaN,NaN,NaN
7,zip,27701,71,False,0.7183,363520.28,0.915
8,zip,27703,536,False,0.9384,378088.83,0.915
9,zip,27704,191,False,0.8796,285295.66,0.915
